# Query Fanout Analysis
### Research Note — AI Brand Visibility Tracker

---

## What is Query Fanout?

When ChatGPT (with web search enabled) receives a user prompt, it does **not** search the web using the verbatim query. Instead, it internally generates one or more *fanout queries* — reformulated searches designed to retrieve the most relevant pages before composing its answer.

**Example:**

| User prompt | OpenAI fanout queries |
|---|---|
| *"What's the best CRM for a Series A startup?"* | `best CRM software for startups 2025` |
| | `HubSpot vs Salesforce for small business` |
| | `CRM tools reviews B2B SaaS` |

These fanout queries directly determine which web pages are retrieved — which in turn influence which brands appear in the response.

## Why Does This Matter for GEO?

If OpenAI consistently fans out to `"[Category] reviews 2025"` or `"[Brand A] vs [Brand B]"`, then:
- **Review sites** (G2, Capterra) and **comparison blogs** have disproportionate influence on which brands get cited
- A brand can increase AI visibility by optimising for these specific query patterns — not just general SEO
- This is the mechanism behind Peec AI's "citation source influence" analysis

## Scope

This notebook analyses fanout queries using **OpenAI only**. Claude (Anthropic) has no web search in its API, and Gemini's Google Search Grounding does not expose explicit search queries. Cross-LLM fanout comparison is therefore not possible with current APIs.

**Data:** 8 prompts × 2–4 fanout queries each ≈ 20–30 fanout queries for analysis.
**Cost:** ~\$0.05–0.10 for the full data collection run.

In [1]:
import os
import sys
import json
import re
import time
import datetime
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from openai import OpenAI
import openai as _oai

print(f"openai SDK version: {_oai.__version__}")
key = os.getenv("OPENAI_API_KEY")
print(f".env loaded: {'yes, key found' if key else 'NO KEY FOUND'}")

client = OpenAI(api_key=key)
assert hasattr(client, "responses"), "Run: pip install -U openai"
print("Responses API available ✓")

CACHE_FILE   = Path("fanout_data.json")
MODEL        = "gpt-4o"
CURRENT_YEAR = str(datetime.date.today().year)


openai SDK version: 2.36.0
.env loaded: yes, key found


Responses API available ✓


In [2]:
# 8 representative prompts — 2 per category — drawn from BATCH_PROMPTS
SAMPLE_PROMPTS = [
    # ── CRM ──────────────────────────────────────────────────────────────
    {
        "id": "crm_01", "category": "crm",
        "text": "What is the best CRM for a Series A startup? Please cite your sources with URLs.",
        "target_brands": ["HubSpot", "Salesforce", "Pipedrive", "Attio", "Zoho CRM", "Close", "Copper"],
    },
    {
        "id": "crm_02", "category": "crm",
        "text": "Which CRM tools are most popular with sales teams at B2B SaaS companies? Please cite your sources with URLs.",
        "target_brands": ["HubSpot", "Salesforce", "Pipedrive", "Attio", "Zoho CRM", "Close", "Copper"],
    },
    # ── Project Management ────────────────────────────────────────────────
    {
        "id": "pm_01", "category": "project_management",
        "text": "What is the best project management tool for a 10-person startup? Please cite your sources with URLs.",
        "target_brands": ["Asana", "Jira", "Linear", "Monday.com", "Notion", "ClickUp", "Trello", "Basecamp"],
    },
    {
        "id": "pm_02", "category": "project_management",
        "text": "Which project management software do engineering teams use most often? Please cite your sources with URLs.",
        "target_brands": ["Asana", "Jira", "Linear", "Monday.com", "Notion", "ClickUp", "Trello", "Basecamp"],
    },
    # ── AI Writing ────────────────────────────────────────────────────────
    {
        "id": "aiw_01", "category": "ai_writing",
        "text": "What are the best AI writing tools for content marketers? Please cite your sources with URLs.",
        "target_brands": ["Jasper", "Copy.ai", "Grammarly", "Writer", "Writesonic", "Notion AI", "Rytr"],
    },
    {
        "id": "aiw_02", "category": "ai_writing",
        "text": "Which AI writing assistants are recommended for long-form content creation? Please cite your sources with URLs.",
        "target_brands": ["Jasper", "Copy.ai", "Grammarly", "Writer", "Writesonic", "Notion AI", "Rytr"],
    },
    # ── Developer Tools ───────────────────────────────────────────────────
    {
        "id": "dev_01", "category": "developer_tools",
        "text": "What project tracking tools do software engineers recommend for agile teams? Please cite your sources with URLs.",
        "target_brands": ["GitHub", "GitLab", "Linear", "Jira", "Shortcut", "Azure DevOps", "YouTrack"],
    },
    {
        "id": "dev_02", "category": "developer_tools",
        "text": "Which issue trackers are most popular with engineering teams at startups? Please cite your sources with URLs.",
        "target_brands": ["GitHub", "GitLab", "Linear", "Jira", "Shortcut", "Azure DevOps", "YouTrack"],
    },
]

print(f"{len(SAMPLE_PROMPTS)} prompts across {len(set(p['category'] for p in SAMPLE_PROMPTS))} categories")

8 prompts across 4 categories


In [3]:
def collect_fanout(prompt: dict, client: OpenAI, model: str = MODEL) -> dict:
    try:
        response = client.responses.create(
            model=model,
            tools=[{"type": "web_search_preview"}],
            input=prompt["text"],
        )

        queries      = []
        response_txt = ""

        for item in response.output:
            if item.type == "web_search_call":
                action = item.action
                if action.type == "search":
                    if action.queries:
                        queries.extend(action.queries)
                    elif action.query:
                        queries.append(action.query)
            elif item.type == "message":
                for block in item.content:
                    if hasattr(block, "text"):
                        response_txt += block.text

        return {
            "id":             prompt["id"],
            "category":       prompt["category"],
            "original":       prompt["text"],
            "target_brands":  prompt["target_brands"],
            "fanout_queries": queries,
            "n_queries":      len(queries),
            "response_text":  response_txt[:600],
            "error":          None,
        }
    except Exception as e:
        return {
            "id":             prompt["id"],
            "category":       prompt["category"],
            "original":       prompt["text"],
            "target_brands":  prompt["target_brands"],
            "fanout_queries": [],
            "n_queries":      0,
            "response_text":  "",
            "error":          str(e),
        }


In [4]:
if CACHE_FILE.exists():
    print(f"Loading cached results from {CACHE_FILE}")
    results = json.loads(CACHE_FILE.read_text())
else:
    print(f"Running {len(SAMPLE_PROMPTS)} prompts via OpenAI Responses API (model={MODEL})...")
    print("Estimated cost: ~$0.05–0.10\n")
    results = []
    for p in SAMPLE_PROMPTS:
        r = collect_fanout(p, client)
        results.append(r)
        if r["error"]:
            print(f"  ✗ [{p['category']}] {p['id']}: {r['error'][:80]}")
        else:
            print(f"  ✓ [{p['category']}] {p['id']}: {r['n_queries']} fanout queries")
            for q in r["fanout_queries"]:
                print(f"      → {q}")
        time.sleep(1)   # avoid rate limits
    CACHE_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False))
    print(f"\nSaved to {CACHE_FILE}")

df  = pd.DataFrame(results)
ok  = df[df["error"].isna()].copy()

print(f"\n{len(ok)}/{len(df)} prompts succeeded")
print(f"Total fanout queries collected: {ok['n_queries'].sum()}")

# Explode into one row per fanout query
rows = []
for _, r in ok.iterrows():
    for q in r["fanout_queries"]:
        rows.append({"id": r["id"], "category": r["category"],
                     "original": r["original"], "fanout_query": q,
                     "target_brands": r["target_brands"]})
qdf = pd.DataFrame(rows)
qdf.head(10)

Running 8 prompts via OpenAI Responses API (model=gpt-4o)...
Estimated cost: ~$0.05–0.10



  ✓ [crm] crm_01: 1 fanout queries
      → best CRM for Series A startup 2023


  ✓ [crm] crm_02: 1 fanout queries
      → most popular CRM tools for B2B SaaS companies 2023


  ✓ [project_management] pm_01: 1 fanout queries
      → best project management tool for 10-person startup 2023


  ✓ [project_management] pm_02: 1 fanout queries
      → most popular project management software for engineering teams 2023


...

  ✓ [ai_writing] aiw_02: 1 fanout queries
      → best AI writing assistants for long-form content 2023


  ✓ [developer_tools] dev_01: 1 fanout queries
      → best project tracking tools for agile teams 2023


  ✓ [developer_tools] dev_02: 1 fanout queries
      → most popular issue trackers for startups 2023



Saved to fanout_data.json

8/8 prompts succeeded
Total fanout queries collected: 8


,id,category,original,fanout_query,target_brands
0,crm_01,crm,What is the best CRM for a Series A startup? P...,best CRM for Series A startup 2023,"[HubSpot, Salesforce, Pipedrive, Attio, Zoho C..."
1,crm_02,crm,Which CRM tools are most popular with sales te...,most popular CRM tools for B2B SaaS companies ...,"[HubSpot, Salesforce, Pipedrive, Attio, Zoho C..."
2,pm_01,project_management,What is the best project management tool for a...,best project management tool for 10-person sta...,"[Asana, Jira, Linear, Monday.com, Notion, Clic..."
3,pm_02,project_management,Which project management software do engineeri...,most popular project management software for e...,"[Asana, Jira, Linear, Monday.com, Notion, Clic..."
4,aiw_01,ai_writing,What are the best AI writing tools for content...,best AI writing tools for content marketers 2023,"[Jasper, Copy.ai, Grammarly, Writer, Writesoni..."
5,aiw_02,ai_writing,Which AI writing assistants are recommended fo...,best AI writing assistants for long-form conte...,"[Jasper, Copy.ai, Grammarly, Writer, Writesoni..."
6,dev_01,developer_tools,What project tracking tools do software engine...,best project tracking tools for agile teams 2023,"[GitHub, GitLab, Linear, Jira, Shortcut, Azure..."
7,dev_02,developer_tools,Which issue trackers are most popular with eng...,most popular issue trackers for startups 2023,"[GitHub, GitLab, Linear, Jira, Shortcut, Azure..."


---
## Section 1 — How Many Fanout Queries Per Prompt?

The first question: how aggressively does OpenAI expand a single user prompt into multiple searches?

In [5]:
print("=== Fanout Count Statistics ===")
print(ok["n_queries"].describe().round(2).to_string())

fig = px.histogram(
    ok, x="n_queries", color="category",
    nbins=6, barmode="stack",
    title="Number of Fanout Queries per Prompt",
    labels={"n_queries": "Fanout queries issued", "count": "Prompts"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", height=350)
fig.show()

print("\nFanout count by category:")
print(ok.groupby("category")["n_queries"].agg(["mean", "min", "max"]).round(1).to_string())

=== Fanout Count Statistics ===
count    8.0
mean     1.0
std      0.0
min      1.0
25%      1.0
50%      1.0
75%      1.0
max      1.0



Fanout count by category:
                    mean  min  max
category                          
ai_writing           1.0    1    1
crm                  1.0    1    1
developer_tools      1.0    1    1
project_management   1.0    1    1


---
## Section 2 — Vocabulary Injection

What words does OpenAI *add* to user prompts when constructing fanout queries? 
We categorise vocabulary into four groups:

| Category | Example words | GEO implication |
|---|---|---|
| **Comparison** | `best`, `vs`, `alternative`, `compared` | Comparison articles rank well |
| **Evaluation** | `review`, `rating`, `recommend`, `pros cons` | Review sites (G2, Capterra) matter |
| **Temporal** | `2024`, `2025` | Recency signals matter for citation |
| **Commercial** | `pricing`, `cost`, `free trial` | Pricing pages may influence results |

In [6]:
KEYWORD_CATS = {
    "comparison":  {"best", "vs", "versus", "alternative", "alternatives",
                    "compared", "comparison", "compare", "top"},
    "evaluation":  {"review", "reviews", "rating", "ratings", "pros", "cons",
                    "recommend", "recommended", "pros cons"},
    "temporal":    {"2024", "2025", CURRENT_YEAR},
    "commercial":  {"pricing", "price", "cost", "free", "trial", "enterprise", "plans"},
}

STOP_WORDS = {
    "the", "a", "an", "is", "are", "for", "to", "in", "of", "and", "or",
    "with", "how", "what", "which", "do", "i", "my", "on", "at", "by",
    "from", "as", "be", "this", "that", "it", "its", "us", "most",
    "more", "use", "using", "used", "their", "they", "you",
}

def word_category(w):
    for cat, kws in KEYWORD_CATS.items():
        if w in kws:
            return cat
    return "other"

# Tokenise all fanout queries
token_rows = []
for _, r in qdf.iterrows():
    tokens = re.findall(r"\b[a-zA-Z0-9]+\b", r["fanout_query"].lower())
    for t in tokens:
        if t not in STOP_WORDS and len(t) > 2:
            token_rows.append({
                "token":    t,
                "category": word_category(t),
                "prompt_category": r["category"],
            })

tdf = pd.DataFrame(token_rows)

# Top 25 words
top = tdf["token"].value_counts().head(25).reset_index()
top.columns = ["word", "count"]
top["category"] = top["word"].apply(word_category)

CAT_COLORS = {
    "comparison": "#F97316",
    "evaluation": "#3B82F6",
    "temporal":   "#8B5CF6",
    "commercial": "#EF4444",
    "other":      "#CBD5E1",
}

fig = px.bar(
    top, x="count", y="word", orientation="h",
    color="category", color_discrete_map=CAT_COLORS,
    title="Top 25 Words in OpenAI Fanout Queries",
    labels={"count": "Occurrences across all fanout queries", "word": ""},
)
fig.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    height=620, yaxis={"categoryorder": "total ascending"},
    legend_title_text="Word category",
)
fig.show()

# Category share
cat_share = (
    tdf[tdf["category"] != "other"]["category"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .rename("share %")
)
print("\nKeyword category share (excluding \"other\"):\n", cat_share.to_string())


Keyword category share (excluding "other"):
 category
comparison    100.0


---
## Section 3 — Does OpenAI Inject Brand Names?

Do fanout queries narrow from a generic category question to brand-specific searches?
We check whether brand names appear in the fanout query that were **not** in the original prompt.

In [7]:
injection_rows = []
for _, r in qdf.iterrows():
    q_lower  = r["fanout_query"].lower()
    or_lower = r["original"].lower()
    brands_in_fanout   = [b for b in r["target_brands"] if b.lower() in q_lower]
    brands_new         = [b for b in brands_in_fanout if b.lower() not in or_lower]
    injection_rows.append({
        "category":         r["category"],
        "original":         r["original"][:70] + "...",
        "fanout_query":     r["fanout_query"],
        "n_brands_fanout":  len(brands_in_fanout),
        "brands_in_fanout": ", ".join(brands_in_fanout) or "—",
        "new_brands":       ", ".join(brands_new) or "—",
        "brand_injected":   len(brands_new) > 0,
    })

bi = pd.DataFrame(injection_rows)

total_q = len(bi)
with_brand    = (bi["n_brands_fanout"] > 0).sum()
new_injection = bi["brand_injected"].sum()

print(f"Total fanout queries: {total_q}")
print(f"Queries containing ≥1 brand name: {with_brand} ({with_brand/total_q*100:.0f}%)")
print(f"Queries with NEW brand names (not in original prompt): {new_injection} ({new_injection/total_q*100:.0f}%)")

print("\n── Fanout queries that injected new brand names ──")
display_cols = ["category", "fanout_query", "new_brands"]
injected = bi[bi["brand_injected"]][display_cols]
if injected.empty:
    print("None — OpenAI kept queries generic (no brand names added).")
else:
    print(injected.to_string(index=False))

Total fanout queries: 8
Queries containing ≥1 brand name: 0 (0%)
Queries with NEW brand names (not in original prompt): 0 (0%)

── Fanout queries that injected new brand names ──
None — OpenAI kept queries generic (no brand names added).


---
## Section 4 — Cross-Reference with Existing Citation Data

**Hypothesis:** if OpenAI's fanout queries frequently contain evaluation keywords (`review`, `rating`, `recommend`), we should see a correspondingly high share of `review_site` citations in the project's existing data for OpenAI.

We load the project's citation data to test this.

In [8]:
try:
    from src.metrics.calculator import citation_by_provider
    from src.storage import store
    store.init()

    cit = citation_by_provider()   # all categories
    if cit.empty:
        print("No citation data in DB — run the batch pipeline first.")
    else:
        # ── % of citations by domain type, per LLM ──────────────────────
        totals = cit.groupby("provider")["count"].transform("sum")
        cit    = cit.copy()
        cit["pct"] = (cit["count"] / totals * 100).round(1)
        cit["provider_label"] = cit["provider"].replace(
            {"openai": "OpenAI", "anthropic": "Anthropic", "gemini": "Gemini"}
        )

        openai_cit = cit[cit["provider"] == "openai"].sort_values("pct", ascending=False)
        print("OpenAI citation distribution:")
        print(openai_cit[["domain_type", "count", "pct"]].to_string(index=False))

        # ── Fanout keyword → citation type connection ────────────────────
        eval_kws  = KEYWORD_CATS["evaluation"]
        comp_kws  = KEYWORD_CATS["comparison"]

        eval_q_pct = qdf["fanout_query"].str.lower().apply(
            lambda q: any(k in q for k in eval_kws)
        ).mean() * 100

        comp_q_pct = qdf["fanout_query"].str.lower().apply(
            lambda q: any(k in q for k in comp_kws)
        ).mean() * 100

        review_site_pct = openai_cit.loc[
            openai_cit["domain_type"] == "review_site", "pct"
        ].values
        tech_media_pct  = openai_cit.loc[
            openai_cit["domain_type"] == "tech_media", "pct"
        ].values

        print(f"\n── Fanout keyword presence ──")
        print(f"  Evaluation keywords in fanout:   {eval_q_pct:.0f}% of queries")
        print(f"  Comparison keywords in fanout:   {comp_q_pct:.0f}% of queries")
        print(f"\n── OpenAI citation outcome ──")
        if len(review_site_pct):
            print(f"  review_site citations:  {review_site_pct[0]:.1f}%")
        if len(tech_media_pct):
            print(f"  tech_media citations:   {tech_media_pct[0]:.1f}%")

        # ── Side-by-side chart ───────────────────────────────────────────
        domain_colors = {
            "review_site":   "#4ADE80", "tech_media":    "#3B82F6",
            "community":     "#F97316", "official_docs": "#8B5CF6",
            "developer":     "#EC4899", "social_media":  "#EAB308",
            "academic":      "#14B8A6", "news":          "#64748B",
            "brand":         "#CBD5E1", "other":         "#94A3B8",
        }
        cit["label"] = cit["pct"].apply(lambda x: f"{x:.0f}%" if x >= 5 else "")
        fig = px.bar(
            cit.sort_values("domain_type"),
            x="provider_label", y="pct", color="domain_type",
            barmode="stack",
            color_discrete_map=domain_colors,
            category_orders={"provider_label": ["OpenAI", "Anthropic", "Gemini"]},
            text="label",
            title="Citation Source Mix by LLM — does fanout vocabulary explain the pattern?",
            labels={"pct": "% of Citations", "provider_label": "", "domain_type": "Source Type"},
        )
        fig.update_traces(textposition="inside", insidetextanchor="middle")
        fig.update_layout(
            plot_bgcolor="white", paper_bgcolor="white", height=420,
            legend=dict(orientation="h", y=-0.25, xanchor="center", x=0.5),
            yaxis=dict(range=[0, 105]),
        )
        fig.show()

except ImportError as e:
    print(f"Could not import project modules: {e}")
    print("Make sure you're running this notebook from the project root.")

OpenAI citation distribution:
domain_type  count   pct
brand_owned    460  85.3
      other     51   9.5
review_site     21   3.9
 tech_media      7   1.3

── Fanout keyword presence ──
  Evaluation keywords in fanout:   0% of queries
  Comparison keywords in fanout:   62% of queries

── OpenAI citation outcome ──
  review_site citations:  3.9%
  tech_media citations:   1.3%


---
## Implications for GEO / Brand Visibility Optimisation

Based on the fanout analysis above, the following content investment priorities follow:

### 1. Review site presence is the highest-leverage action
If evaluation keywords (`review`, `rating`, `recommend`) appear in a large share of fanout queries, OpenAI is systematically retrieving review-aggregator pages before composing its answer. A brand without G2, Capterra, or Trustpilot entries is effectively invisible at the retrieval stage — regardless of product quality.

### 2. Comparison articles create structural advantage
Comparison keywords (`best`, `vs`, `alternative`) in fanout queries signal that OpenAI is looking for head-to-head articles. A brand that appears prominently in `"HubSpot alternatives"` or `"best CRM for startups"` listicles will be retrieved and cited even when the user did not mention it by name.

### 3. Year injection means recency matters
If OpenAI consistently appends the current year (e.g., `best CRM 2025`), articles published or updated recently have a structural advantage over older content — even if the older content is more comprehensive.

### 4. Brand name injection reveals prioritisation logic
When OpenAI injects specific brand names into fanout queries (brands not mentioned in the user's original prompt), it reveals its prior beliefs about which brands are relevant. This is likely derived from training data frequency, not web search results — and is difficult to influence in the short term.

### 5. This analysis is OpenAI-specific
Claude and Gemini do not expose their retrieval queries via API. The fanout patterns described here apply to OpenAI only. Citation pattern differences observed in the main dashboard's LLM Behavior Analysis tab may partly reflect different fanout vocabularies — but this cannot be confirmed without equivalent API access to the other models.

---

*This notebook is a research note within the [AI Brand Visibility Tracker](https://github.com/jeannineshiu/ai-brand-visibility-tracker) project.*